## 시나리오

데이터 분석은 파이썬에서, 결과물은 HWP 표로 — 가장 자주 쓰는 자동화 패턴입니다.

- pandas DataFrame
- CSV 파일
- list of dict (API 응답 등)

모두 같은 헬퍼 함수로 HWP 표로 변환합니다.

## 범용 헬퍼

```python
from hwpapi.core import App


def insert_table(app, rows: list[list], headers: list[str] = None,
                 bold_header: bool = True,
                 header_align: int = 3):   # 3=center
    """
    app에 현재 커서 위치부터 표를 삽입합니다.
    rows: [[c1, c2, ...], [c1, c2, ...], ...]
    headers: 있으면 첫 행에 삽입되고 굵게 표시됨
    """
    if not rows:
        return

    # 크기 결정
    n_cols = max(len(r) for r in rows)
    if headers:
        n_cols = max(n_cols, len(headers))
    n_rows = len(rows) + (1 if headers else 0)

    # 표 생성
    action = app.actions.TableCreate
    action.pset.Rows = n_rows
    action.pset.Cols = n_cols
    action.run()

    # 헤더
    if headers:
        if bold_header:
            app.set_charshape(bold=True)
            app.set_parashape(align_type=header_align)
        for h in headers:
            app.insert_text(str(h))
            app.api.Run("TableRightCell")
        # 헤더 끝난 뒤 서식 해제
        if bold_header:
            app.set_charshape(bold=False)
            app.set_parashape(align_type=1)

    # 데이터 행
    for row in rows:
        for i in range(n_cols):
            val = row[i] if i < len(row) else ""
            app.insert_text(str(val))
            app.api.Run("TableRightCell")
```

## 예 1: pandas DataFrame

```python
import pandas as pd

df = pd.read_csv("sales.csv")
# 또는 분석 결과
summary = df.groupby("region")["revenue"].agg(["sum", "mean"]).reset_index()

app = App(is_visible=True)
app.insert_text("2026년 1분기 지역별 매출 요약\n\n")

insert_table(
    app,
    rows=summary.values.tolist(),
    headers=summary.columns.tolist(),
)
```

## 예 2: CSV 파일 직접

```python
import csv

with open("employees.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    rows = list(reader)

headers, data = rows[0], rows[1:]

app = App(is_visible=True)
insert_table(app, rows=data, headers=headers)
```

## 예 3: API 응답 (list of dict)

```python
# 서버에서 받아온 데이터
users = [
    {"id": 1, "name": "김철수", "email": "kim@co.kr", "role": "Admin"},
    {"id": 2, "name": "이영희", "email": "lee@co.kr", "role": "User"},
    {"id": 3, "name": "박민수", "email": "park@co.kr", "role": "User"},
]

headers = list(users[0].keys())
rows = [[u[k] for k in headers] for u in users]

app = App(is_visible=True)
app.insert_text("사용자 목록\n\n")
insert_table(app, rows=rows, headers=headers)
```

## 숫자 포맷팅

파이썬 측에서 먼저 문자열로 포맷팅한 뒤 표에 넣는 것이 가장 간단합니다.

```python
def fmt_money(v):
    return f"{v:,.0f}원"

def fmt_percent(v):
    return f"{v * 100:.1f}%"

rows = [
    [name, fmt_money(sales), fmt_percent(share)]
    for name, sales, share in raw_data
]
insert_table(app, rows=rows, headers=["제품", "매출", "점유율"])
```

## 표 이후 본문 이어쓰기

표 생성 후 커서가 표 안에 있으므로, 표 다음에 글을 이어 쓰려면 표 밖으로 나와야 합니다.

```python
insert_table(app, rows, headers)
app.move.bottom_of_file()  # 문서 끝으로
app.insert_text("\n표에서 보시듯이...\n")
```

## 📄 실행 결과 문서

**예 1 (pandas DataFrame)** — 다음과 같은 HWP 문서가 생성됩니다:

<div class="preview-doc">

<div style="font-size:1.3rem;font-weight:700;margin:1rem 0;">2026년 1분기 지역별 매출 요약</div>

<table style="border-collapse:collapse;margin:1rem 0;">
<thead><tr>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">region</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">sum</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">mean</th>
</tr></thead>
<tbody>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">East</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">1,850,000</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">185,000</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">North</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">1,320,000</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">132,000</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">South</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">620,000</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">62,000</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">West</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">440,000</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">44,000</td></tr>
</tbody>
</table>


</div>**예 3 (API 응답 list of dict):**

<div class="preview-doc">

<div style="font-size:1.3rem;font-weight:700;margin:1rem 0;">사용자 목록</div>

<table style="border-collapse:collapse;margin:1rem 0;">
<thead><tr>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">id</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">name</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">email</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">role</th>
</tr></thead>
<tbody>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">1</td><td style="border:1px solid #888;padding:0.4rem 1rem;">김철수</td><td style="border:1px solid #888;padding:0.4rem 1rem;">kim@co.kr</td><td style="border:1px solid #888;padding:0.4rem 1rem;">Admin</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">2</td><td style="border:1px solid #888;padding:0.4rem 1rem;">이영희</td><td style="border:1px solid #888;padding:0.4rem 1rem;">lee@co.kr</td><td style="border:1px solid #888;padding:0.4rem 1rem;">User</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">3</td><td style="border:1px solid #888;padding:0.4rem 1rem;">박민수</td><td style="border:1px solid #888;padding:0.4rem 1rem;">park@co.kr</td><td style="border:1px solid #888;padding:0.4rem 1rem;">User</td></tr>
</tbody>
</table>


**숫자 포맷팅 예 — `제품 / 매출 / 점유율`:**

<div class="preview-doc">

<table style="border-collapse:collapse;margin:1rem 0;">
<thead><tr>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">제품</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">매출</th>
<th style="border:1px solid #555;padding:0.5rem 1rem;background:#2c3e50;color:white;">점유율</th>
</tr></thead>
<tbody>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">A</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">1,200,000원</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">52.5%</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">B</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">800,000원</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">35.0%</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1rem;">C</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">285,000원</td><td style="border:1px solid #888;padding:0.4rem 1rem;text-align:right;">12.5%</td></tr>
</tbody>
</table>
